# Classificatore esplorativo dei temi politici

Questo notebook permette di **leggere gli output esistenti** e comprenderli senza usare il terminale. La modalità predefinita è di sola lettura: non riaddestra il modello e non sovrascrive file.

La logica riutilizzabile vive in `src/news_topic_model.py`. Il notebook si limita a configurarla e richiamarla: **TF-IDF** rappresenta quanto un termine caratterizza un articolo rispetto al corpus; **NMF** individua configurazioni lessicali ricorrenti. L'output è un insieme di microtemi emergenti da validare, non una tassonomia definitiva.

Il corpus Media Cloud descrive come partiti e leader sono raccontati negli articoli raccolti dalle query. Non misura automaticamente comunicazione diretta, attività istituzionale, consenso, sentiment verso un partito o stance su una policy.

## Modalità disponibili

1. **Normale (predefinita)** — carica `data/processed/news_topic_review.csv`, `news_topic_terms.csv` e `topic_model_metadata.json` senza modificarli.
2. **Demo** — usa un piccolo corpus sintetico e una cartella temporanea, eliminata al termine della cella. Non produce risultati di progetto.
3. **Run completo del classificatore** — legge il corpus locale completo e scrive in una nuova directory datata. Richiede sia il flag sia una frase di consenso esatta. Non esegue discovery o download degli articoli.

Google Drive non è mai usato come input. Il mirror degli output è separato, opzionale e disattivato.

In [ ]:
import json
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

def find_repository_root(start):
    """Trova la root senza assumere il computer o la cartella dell'utente."""
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'news_topic_model.py').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Root del repository non trovata: aprire questo notebook dal progetto temi-politici.')

ROOT = find_repository_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_SCRIPT = ROOT / 'src' / 'news_topic_model.py'
AUDIT_RUNNER = ROOT / 'scripts' / 'run_topic_audit.py'
CANONICAL_AUDIT_CONFIG = ROOT / 'config' / 'topic_audit.json'
FULLTEXT_FILE = ROOT / 'data' / 'raw' / 'mediacloud_fulltext.jsonl'
DEFAULT_OUTPUT_DIR = ROOT / 'data' / 'processed'
CANONICAL_AUDIT_DIR = ROOT / 'reports' / 'topic_audit'

print(f'Repository: {ROOT}')
print(f'Python del kernel: {sys.executable}')

## Configurazione sicura

Lasciare entrambi i flag su `False` per la modalità normale. Attivare al massimo una modalità. Il consenso testuale impedisce di avviare per errore un'elaborazione completa o una copia su Drive.

In [ ]:
RUN_FULL_PIPELINE = False
RUN_DEMO = False
FULL_PIPELINE_CONSENT = ''  # Per il run completo: 'CONFERMO RUN COMPLETO'

MIRROR_TO_DRIVE = False
DRIVE_MIRROR_CONSENT = ''  # Per il mirror: 'CONFERMO MIRROR DRIVE'

N_TOPICS = 12
MIN_DF = 3
MAX_DF = 0.85
MIN_TEXT_CHARS = 300
RANDOM_STATE = 42

if RUN_FULL_PIPELINE and RUN_DEMO:
    raise ValueError('Scegliere una sola modalità: run completo oppure demo.')

## Preflight

Questa cella controlla soltanto la presenza degli input e degli output locali. Non apre Google Drive, non crea directory e non modifica file.

In [ ]:
REVIEW_FILE = DEFAULT_OUTPUT_DIR / 'news_topic_review.csv'
TERMS_FILE = DEFAULT_OUTPUT_DIR / 'news_topic_terms.csv'
METADATA_FILE = DEFAULT_OUTPUT_DIR / 'topic_model_metadata.json'

def describe(path):
    if not path.is_file():
        return 'assente'
    return f'presente ({path.stat().st_size / 1_000_000:.2f} MB)'

for label, path in [
    ('Corpus full-text', FULLTEXT_FILE),
    ('Revisione articoli', REVIEW_FILE),
    ('Termini dei topic', TERMS_FILE),
    ('Metadati del modello', METADATA_FILE),
]:
    print(f'{label}: {describe(path)}')

print('Mirror Drive:', 'richiesto' if MIRROR_TO_DRIVE else 'disattivato')

## Funzione di orchestrazione

Le funzioni seguenti non implementano TF-IDF, NMF o calcoli di audit: richiamano `src/news_topic_model.py` e il runner ufficiale `scripts/run_topic_audit.py` con il Python del kernel, mostrando il log nel notebook.

In [ ]:
def run_classifier(input_path, output_dir, n_topics, min_df, max_df, min_text_chars, random_state):
    command = [
        sys.executable, str(MODEL_SCRIPT),
        '--input', str(input_path),
        '--output-dir', str(output_dir),
        '--n-topics', str(n_topics),
        '--min-df', str(min_df),
        '--max-df', str(max_df),
        '--min-text-chars', str(min_text_chars),
        '--random-state', str(random_state),
    ]
    print('Esecuzione della logica salvata in:', MODEL_SCRIPT.relative_to(ROOT))
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError(f'Classificatore terminato con codice {completed.returncode}.')
    return Path(output_dir)

def run_topic_audit(config_path):
    config_path = Path(config_path).resolve()
    config_relative = config_path.relative_to(ROOT).as_posix()
    command = [sys.executable, str(AUDIT_RUNNER), '--config', config_relative]
    print('Esecuzione del runner ufficiale:', AUDIT_RUNNER.relative_to(ROOT))
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError(f'Audit terminato con codice {completed.returncode}.')

## Run completo del classificatore — protetto

Per avviarlo servono **due azioni esplicite** nella configurazione: `RUN_FULL_PIPELINE = True` e `FULL_PIPELINE_CONSENT = 'CONFERMO RUN COMPLETO'`. Il classificatore e il successivo audit riproducibile sono scritti in una nuova sottocartella di `data/processed/topic_model_runs/`; se esiste già, il run si ferma. Gli output predefiniti e `reports/topic_audit/` non vengono sovrascritti.

Questo run può richiedere tempo e memoria. Non acquisisce nuovi articoli e non usa Google Drive come sorgente.

In [ ]:
ACTIVE_OUTPUT_DIR = DEFAULT_OUTPUT_DIR
ACTIVE_AUDIT_DIR = CANONICAL_AUDIT_DIR

if RUN_FULL_PIPELINE:
    if FULL_PIPELINE_CONSENT != 'CONFERMO RUN COMPLETO':
        raise PermissionError('Run non autorizzato: inserire la frase di consenso esatta.')
    if not FULLTEXT_FILE.is_file():
        raise FileNotFoundError(f'Corpus full-text non trovato: {FULLTEXT_FILE.relative_to(ROOT)}')
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    requested_output_dir = ROOT / 'data' / 'processed' / 'topic_model_runs' / f'run_{run_id}'
    if requested_output_dir.exists():
        raise FileExistsError(f'Output già esistente, nessuna sovrascrittura: {requested_output_dir}')
    ACTIVE_OUTPUT_DIR = run_classifier(
        FULLTEXT_FILE, requested_output_dir, N_TOPICS, MIN_DF, MAX_DF,
        MIN_TEXT_CHARS, RANDOM_STATE
    )
    ACTIVE_AUDIT_DIR = requested_output_dir / 'audit'
    if ACTIVE_AUDIT_DIR.exists():
        raise FileExistsError(f'Audit già esistente, nessuna sovrascrittura: {ACTIVE_AUDIT_DIR}')
    audit_config = json.loads(CANONICAL_AUDIT_CONFIG.read_text(encoding='utf-8'))
    audit_config.update({
        'input_review_csv': (ACTIVE_OUTPUT_DIR / 'news_topic_review.csv').relative_to(ROOT).as_posix(),
        'input_metadata_json': (ACTIVE_OUTPUT_DIR / 'topic_model_metadata.json').relative_to(ROOT).as_posix(),
        'output_dir': ACTIVE_AUDIT_DIR.relative_to(ROOT).as_posix(),
    })
    audit_config_path = ACTIVE_OUTPUT_DIR / 'topic_audit_config.json'
    if audit_config_path.exists():
        raise FileExistsError(f'Configurazione audit già esistente: {audit_config_path}')
    audit_config_path.write_text(
        json.dumps(audit_config, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    run_topic_audit(audit_config_path)
else:
    print('Run completo non avviato. Modalità sicura attiva.')

## Caricamento degli output esistenti

In modalità normale questa sezione legge gli output canonici. Dopo un run completo autorizzato legge invece la nuova directory del run. Vengono caricate dal file articoli soltanto le colonne utili alla revisione, senza alterare i CSV.

In [ ]:
def load_outputs(output_dir):
    paths = {
        'review': Path(output_dir) / 'news_topic_review.csv',
        'terms': Path(output_dir) / 'news_topic_terms.csv',
        'metadata': Path(output_dir) / 'topic_model_metadata.json',
    }
    missing = [path for path in paths.values() if not path.is_file()]
    if missing:
        relative = [str(path.relative_to(ROOT)) for path in missing]
        raise FileNotFoundError(f'Output mancanti: {relative}')
    terms = pd.read_csv(paths['terms'], encoding='utf-8-sig', keep_default_na=False)
    review = pd.read_csv(
        paths['review'], encoding='utf-8-sig',
        usecols=['topic_id', 'confidenza_topic', 'domain', 'title']
    )
    metadata = json.loads(paths['metadata'].read_text(encoding='utf-8'))
    return review, terms, metadata

if not RUN_DEMO:
    review, terms, metadata = load_outputs(ACTIVE_OUTPUT_DIR)
    print(f'Output letti da: {ACTIVE_OUTPUT_DIR.relative_to(ROOT)}')
    print(f"Articoli dichiarati nei metadati: {metadata.get('articles', 'non disponibile')}")
    print(f"Topic: {metadata.get('n_topics', len(terms))}; seed: {metadata.get('random_state', 'non disponibile')}")
    display(terms[['topic_id', 'articoli_dominanti', 'termini_caratteristici', 'macrotema_validato', 'note_revisione']])
else:
    print('Caricamento normale saltato: modalità demo selezionata.')

## Audit riproducibile — lettura dei report

In modalità normale, questa sezione legge in sola lettura i report canonici presenti in `reports/topic_audit/`. Dopo un futuro run completo autorizzato, legge invece l'audit separato del nuovo run. Ogni file è gestito indipendentemente: un report mancante o non leggibile produce un messaggio chiaro senza interrompere le altre visualizzazioni.

Le tabelle descrivono distribuzione, nettezza delle assegnazioni, concentrazione per dominio e possibili duplicazioni. Sono controlli quantitativi: non assegnano etichette semantiche e non trasformano automaticamente i topic in macrotemi validati.

In [ ]:
def read_audit_csv(path, label):
    try:
        frame = pd.read_csv(path)
    except FileNotFoundError:
        print(f'{label}: report non presente ({path.relative_to(ROOT)}).')
        return None
    except (OSError, ValueError, pd.errors.ParserError) as error:
        print(f'{label}: report non leggibile ({type(error).__name__}: {error}).')
        return None
    print(f'{label}: {path.relative_to(ROOT)}')
    return frame

def read_audit_json(path, label):
    try:
        value = json.loads(path.read_text(encoding='utf-8'))
    except FileNotFoundError:
        print(f'{label}: report non presente ({path.relative_to(ROOT)}).')
        return None
    except (OSError, UnicodeError, json.JSONDecodeError) as error:
        print(f'{label}: report non leggibile ({type(error).__name__}: {error}).')
        return None
    print(f'{label}: {path.relative_to(ROOT)}')
    return value

if not RUN_DEMO:
    topic_distribution = read_audit_csv(
        ACTIVE_AUDIT_DIR / 'topic_distribution.csv', 'Distribuzione dei topic'
    )
    if topic_distribution is not None:
        display(topic_distribution)

    confidence_summary = read_audit_csv(
        ACTIVE_AUDIT_DIR / 'confidence_summary.csv', 'Sintesi della confidenza'
    )
    if confidence_summary is not None:
        display(confidence_summary[confidence_summary['variable'].eq('confidenza_topic')])

    domain_summary = read_audit_csv(
        ACTIVE_AUDIT_DIR / 'domain_summary.csv', 'Domini dominanti per topic'
    )
    if domain_summary is not None:
        display(domain_summary[domain_summary['rank'].eq(1)])

    duplicate_summary = read_audit_json(
        ACTIVE_AUDIT_DIR / 'duplicate_summary.json', 'Duplicati e quasi duplicati'
    )
    if duplicate_summary is not None:
        duplicate_rows = []
        for section in ('exact_excerpt', 'exact_title_excerpt', 'near_duplicate_proxy'):
            for metric, value in duplicate_summary.get(section, {}).items():
                duplicate_rows.append({'sezione': section, 'metrica': metric, 'valore': value})
        display(pd.DataFrame(duplicate_rows))
else:
    print('Report di audit reali non caricati in modalità demo.')

## Lettura guidata

La **confidenza normalizzata** è il peso del topic dominante diviso per la somma dei pesi dell'articolo. Aiuta a trovare esempi più netti, ma non è una probabilità calibrata. La tabella seguente mostra al massimo cinque titoli ad alta confidenza per topic: serve come punto di partenza per il controllo umano, non dimostra da sola la coerenza politica del topic.

In [ ]:
if not RUN_DEMO:
    representative = (
        review.sort_values(['topic_id', 'confidenza_topic'], ascending=[True, False])
        .groupby('topic_id', as_index=False, group_keys=False)
        .head(5)
    )
    display(representative[['topic_id', 'confidenza_topic', 'domain', 'title']])
else:
    print('Anteprima degli output reali saltata in modalità demo.')

## Demo separata e non distruttiva

Con `RUN_DEMO = True`, la cella crea dodici record sintetici in una directory temporanea e richiama lo stesso script in `src/`. Legge i risultati prima di eliminare automaticamente la directory. La demo verifica il flusso tecnico: i topic ottenuti **non sono risultati del progetto**.

In [ ]:
if RUN_DEMO:
    demo_themes = [
        ('lavoro', 'Il dibattito riguarda lavoro, salari, contratti, occupazione e tutela dei lavoratori.'),
        ('sanita', 'Il confronto riguarda sanità, ospedali, medici, assistenza e liste di attesa.'),
        ('ambiente', 'La discussione riguarda ambiente, energia, clima, emissioni e transizione ecologica.'),
    ]
    demo_rows = []
    for topic, text in demo_themes:
        for index in range(4):
            demo_rows.append({
                'url': f'https://example.invalid/{topic}/{index}',
                'domain': 'example.invalid',
                'seendate': f'2026-01-{index + 1:02d}',
                'title': f'Esempio sintetico su {topic} {index + 1}',
                'text': f'{text} {text} Caso sintetico numero {index + 1}.',
                'language': 'it',
            })
    with tempfile.TemporaryDirectory(prefix='temi_politici_demo_') as temporary_dir:
        temporary_dir = Path(temporary_dir)
        demo_input = temporary_dir / 'demo.jsonl'
        demo_output = temporary_dir / 'output'
        pd.DataFrame(demo_rows).to_json(demo_input, orient='records', lines=True, force_ascii=False)
        run_classifier(demo_input, demo_output, 3, 1, 1.0, 50, RANDOM_STATE)
        demo_review, demo_terms, demo_metadata = load_outputs(demo_output)
        display(demo_terms[['topic_id', 'articoli_dominanti', 'termini_caratteristici']])
        print(f"Demo completata su {demo_metadata['articles']} record sintetici.")
    print('Directory temporanea eliminata: nessun output di progetto modificato.')
else:
    print('Demo non avviata. Impostare RUN_DEMO = True per eseguirla.')

## Mirror Google Drive — opzionale

Il calcolo usa sempre input e output locali al repository. Solo dopo un run autorizzato, gli output possono essere copiati nel mirror configurato da `src/drive_mirror.py`. Per evitare copie accidentali, servono `MIRROR_TO_DRIVE = True` e la frase esatta `CONFERMO MIRROR DRIVE`. In modalità normale il mirror non parte.

In [ ]:
if MIRROR_TO_DRIVE:
    if not RUN_FULL_PIPELINE:
        raise PermissionError('Il mirror è consentito solo dopo un run completo autorizzato in questa sessione.')
    if DRIVE_MIRROR_CONSENT != 'CONFERMO MIRROR DRIVE':
        raise PermissionError('Mirror non autorizzato: inserire la frase di consenso esatta.')
    from src.drive_mirror import mirror_file
    for filename in ('news_topic_review.csv', 'news_topic_terms.csv', 'topic_model_metadata.json'):
        mirror_file(ACTIVE_OUTPUT_DIR / filename, section=f'topic_model_runs/{ACTIVE_OUTPUT_DIR.name}')
else:
    print('Mirror Google Drive non eseguito.')

## Human check richiesto

Per ogni `topic_id`, controllare i termini caratteristici e 5–10 articoli rappresentativi, annotando: coerenza politica, eventuale boilerplate, fonti prevalenti, casi ambigui ed etichetta provvisoria. `macrotema_validato` e `note_revisione` non vengono compilati automaticamente dal notebook.

Un topic NMF è un **microtema emergente**. La sua frequenza descrive salienza nel corpus raccolto, non consenso, sentiment, stance o comportamento istituzionale. L'aggregazione in macrotemi candidati richiede una fase successiva e validazione umana.